# 04 — Audio embeddings subset (Phase 2, Step 2)

Качаем `embeddings.parquet` (14 GB, 7.72M items) с HF в кэш `huggingface_hub`,
фильтруем по `item_id_to_idx.pkl` из Phase 1 (276,305 items), сохраняем
`[n_items+1, 128]` float32 в `artifacts/audio/embeddings.npy` (~135 MB).

Запускать на **Colab** (95 GB RAM).

In [ ]:
# Colab bootstrap (раскомментировать в Colab):
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -b models-1 https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q pyarrow huggingface_hub numpy

In [ ]:
import sys, pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

with open(PROJECT_ROOT / 'artifacts' / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)
print(f'items: {len(item_id_to_idx):,}, max idx: {max(item_id_to_idx.values()):,}')

In [ ]:
from src.data.audio_embeddings import extract_audio_subset

OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'audio' / 'embeddings.npy'
embeds = extract_audio_subset(item_id_to_idx, OUTPUT_PATH, use_normalized=False)

In [ ]:
import numpy as np

arr = np.load(OUTPUT_PATH)
norms = np.linalg.norm(arr[1:], axis=1)
print('shape:', arr.shape, 'dtype:', arr.dtype)
print('PAD row zero:', not arr[0].any())
print(f'rows with zero norm: {int((norms == 0).sum())} / {arr.shape[0] - 1}')
print(f'norm (non-zero): mean={norms[norms > 0].mean():.4f}, '
      f'min={norms[norms > 0].min():.4f}, max={norms[norms > 0].max():.4f}')
print(f'NaN: {int(np.isnan(arr).sum())}, Inf: {int(np.isinf(arr).sum())}')
print(f'file size: {OUTPUT_PATH.stat().st_size / 2**20:.1f} MB')